# RunKeeper GPX tracks → `prod.bronze.raw_runkeeper_gpx`

Lands RunKeeper's GPS tracks, one row per file:

```
/Volumes/prod/bronze/stage/runkeeper/gpx_maps/2026-09-28-200444.gpx   one track per activity, named for its start
```

- `gpx_file` is the file's name (the primary key); `gpx` is the whole file, as text.
- `activity_start` is the start time in the file's name, written as the activity log writes its `Date`
  (`2026-09-28-200444.gpx` → `2026-09-28 20:04:44`). The app matches a track to its activity by the log's
  **GPX File** column, else by that start time.
- `track_started_at` is the first `<time>` in the file (UTC), and `points` counts its track points: enough to
  check a file without parsing it again.
- **Only new or changed files are written** (MERGE on `gpx_file`, compared by name and size), each with a fresh
  `_ingested_at`. A file is read only when it's written. If the same name sits in two folders, the newest wins.
- The table has **Change Data Feed** on and `gpx_file` as its **primary key**, as a Lakebase synced table needs.

The activity log is landed by the companion notebook, `runkeeper_activities_to_bronze`.

**Parameters** (widgets, or the job task's parameters): `gpx_dir`, `table`, `dry_run` (`true` counts what would be
written and writes nothing).

In [ ]:
import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

GPX_DIR = "/Volumes/prod/bronze/stage/runkeeper/gpx_maps"
TABLE = "prod.bronze.raw_runkeeper_gpx"
KEY = "gpx_file"

COLUMNS = ["gpx_file", "activity_start", "track_started_at", "points", "gpx", "gpx_bytes",
           "_source_file", "_file_modified_at", "_row_hash", "_ingested_at"]
TYPES = {"points": "long", "gpx_bytes": "long", "_file_modified_at": "timestamp", "_ingested_at": "timestamp"}
DDL = """CREATE TABLE IF NOT EXISTS {table} (
    gpx_file STRING NOT NULL COMMENT 'The file name, e.g. 2026-09-28-200444.gpx',
    activity_start STRING COMMENT 'The start time in the file name, as the activity log writes Date',
    track_started_at STRING COMMENT 'The first time in the track (UTC, as the file writes it)',
    points BIGINT COMMENT 'Track points in the file',
    gpx STRING COMMENT 'The whole GPX file, as text',
    gpx_bytes BIGINT,
    _source_file STRING,
    _file_modified_at TIMESTAMP,
    _row_hash STRING COMMENT 'The file name and size it was written from; an unchanged file is not written again',
    _ingested_at TIMESTAMP COMMENT 'When this row was last written',
    CONSTRAINT raw_runkeeper_gpx_pk PRIMARY KEY (gpx_file)
) USING DELTA
COMMENT 'RunKeeper GPX tracks (gpx_maps); landed by the runkeeper_gpx_to_bronze notebook'
TBLPROPERTIES (delta.enableChangeDataFeed = true)"""

In [ ]:
def _files(folder, suffix):
    """Every file under folder (any depth) whose name ends with suffix, case aside: [(path, size, modified_at)]."""
    out = []
    for root, _dirs, names in os.walk(folder):
        for name in names:
            if name.lower().endswith(suffix):
                path = os.path.join(root, name)
                st = os.stat(path)
                out.append((path, st.st_size, datetime.fromtimestamp(st.st_mtime, tz=timezone.utc)))
    return out


def existing_hashes(spark, table, key, create=True):
    """{key: _row_hash} already landed; the table is made first unless this is a dry run."""
    if create:
        spark.sql(DDL.format(table=table))
    elif not spark.catalog.tableExists(table):
        return {}
    return {r[key]: r["_row_hash"] for r in spark.table(table).select(key, "_row_hash").collect()}


def merge(spark, table, key, rows, prepare=lambda r: r, batch=200):
    """Upsert rows on key, a batch at a time; prepare(row) gives the row as written (e.g. reads its file)."""
    from pyspark.sql import functions as F
    from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType
    types = {"long": LongType(), "timestamp": TimestampType()}
    schema = StructType([StructField(c, types.get(TYPES.get(c), StringType())) for c in COLUMNS if c != "_ingested_at"])
    for i in range(0, len(rows), batch):
        part = [prepare(r) for r in rows[i:i + batch]]
        df = spark.createDataFrame([[r.get(f.name) for f in schema.fields] for r in part], schema)
        df.withColumn("_ingested_at", F.current_timestamp()).createOrReplaceTempView("runkeeper_batch")
        spark.sql(f"""MERGE INTO {table} t USING runkeeper_batch s ON t.{key} = s.{key}
                      WHEN MATCHED AND t._row_hash <> s._row_hash THEN UPDATE SET *
                      WHEN NOT MATCHED THEN INSERT *""")


def setting(name, default, label):
    """A notebook widget (a job task parameter overrides it), or outside Databricks an environment variable."""
    try:
        dbutils.widgets.text(name, default, label)              # noqa: F821  (Databricks provides dbutils)
        return dbutils.widgets.get(name) or default             # noqa: F821
    except NameError:
        return os.environ.get(name.upper(), default)

NAMED = re.compile(r"(\d{4}-\d{2}-\d{2})-(\d{2})(\d{2})(\d{2})")


def activity_start(name):
    """The start time in a RunKeeper file name: '2026-09-28-200444.gpx' → '2026-09-28 20:04:44'."""
    m = NAMED.search(name or "")
    if not m:
        return None
    text = f"{m.group(1)} {m.group(2)}:{m.group(3)}:{m.group(4)}"
    try:
        datetime.strptime(text, "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None
    return text


def read_files(folder):
    """GPX files under folder, one per name (the newest file winning): [{gpx_file, …}]."""
    latest = {}
    for path, size, modified_at in sorted(_files(folder, ".gpx"), key=lambda f: (f[2], f[0])):
        name = os.path.basename(path)
        latest[name] = {"gpx_file": name, "activity_start": activity_start(name), "gpx_bytes": size,
                        "_source_file": path, "_file_modified_at": modified_at}
    return list(latest.values())


def row_hash(row):
    return hashlib.sha256(json.dumps([row["gpx_file"], row["gpx_bytes"]]).encode()).hexdigest()


def plan(files, existing):
    """The files to write (new or changed) and a summary. existing: {gpx_file: _row_hash} already landed."""
    write = [{**f, "_row_hash": row_hash(f)} for f in files]
    write = [f for f in write if existing.get(f["gpx_file"]) != f["_row_hash"]]
    return write, {"gpx_files": len(files), "without_start_in_name": sum(1 for f in files if not f["activity_start"]),
                   "new_or_changed": len(write), "unchanged": len(files) - len(write)}


def with_track(row):
    """The row as written: the file's text, its first time and how many points it has."""
    text = Path(row["_source_file"]).read_text(encoding="utf-8", errors="replace")
    first = re.search(r"<trkpt\b.*?<time>\s*([^<\s]+)\s*</time>", text, re.S)
    return {**row, "gpx": text, "points": len(re.findall(r"<trkpt\b", text)), "track_started_at": first and first.group(1)}


def run(spark, gpx_dir=GPX_DIR, table=TABLE, dry_run=False):
    files = read_files(gpx_dir)
    rows, summary = plan(files, existing_hashes(spark, table, KEY, create=not dry_run))
    if rows and not dry_run:
        merge(spark, table, KEY, rows, prepare=with_track)
    return {"table": table, "gpx_dir": gpx_dir, "dry_run": dry_run, **summary}

## Run

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
summary = run(spark, setting("gpx_dir", GPX_DIR, "GPX folder"), setting("table", TABLE, "Bronze table"), dry_run=setting("dry_run", "false", "Dry run (true: count only)").lower() == "true")
print(json.dumps(summary, indent=2))
try:
    dbutils.notebook.exit(json.dumps(summary))                   # noqa: F821  (the job run's output)
except NameError:
    pass